<h1 id="Planout-A/B-test-results">Planout A/B test results</h1>

<nav aria-label="Report contents" id="report-contents"><h2 id="toc">Contents</h2><ul>
<li><a href="#CONFIG">Config</a></li>
<li><a href="#KPI-Metrics">KPI Metrics</a></li>
<li><a href="#Appendix">Appendix</a></li>
<li><a href="#New-Users">New Users</a></li>
<li><a href="#Existing-Users">Existing Users</a></li>
<li><a href="#Variant-Count">Variant Count</a></li>
<li><a href="#Total-active-days">Total active days</a></li>
<li><a href="#L7">L7</a></li>
<li><a href="#Posts">Post</a></li>
<li><a href="#Reblogs">Reblogs</a></li>
<li><a href="#Likes">Likes</a></li>
<li><a href="#Replies">Replies</a></li>
<li><a href="#Messages">Messages</a></li>
<li><a href="#Follows">Follows</a></li>
<li><a href="#Unfollows">Unfollows</a></li>
<li><a href="#Algo-Dash-Push">Algo push</a></li>
<li><a href="#Transactional-Push">Transactional push</a></li>
<li><a href="#CRM-Push">CRM push</a></li>
</ul></nav>

In [1]:
import os

import pandas as pd 
from IPython.display import HTML

from planout.template_query import (hive_or_pickle, get_base_mapping, get_results_parallel, get_pickle_name)
from planout.template import (style, calc, calc_format, toggle_html, summary, change_name)
from planout.template_helper import filter_df
from planout.sql import helper
import time
import datetime
from datatools.db import hive

HTML(toggle_html)


Show / hide code

<h2 id="CONFIG">CONFIG</h2>

In [2]:
options = {

    'dt_start': '2018-04-12',  
    'time_start': '16:00:00',
    'dt_end': '2018-05-16',
    'time_end': '10:00:00',
    
    'experiment_name': 'android_app_icon_badging',
    'experiment_id': 717,
    'experiment_checksum': '2cacd83b',
    
    'variants': {
        'control': 'control', 
        'test1': 'enabled'
    },
    'mobile_only': False,
    'android_only': False,
    'ios_only': False
}

# make a directory to pickle files
directory_name = options['experiment_name']
if not os.path.exists(directory_name):
    os.makedirs(directory_name)
    
df_mapping = get_base_mapping(**options)


In [4]:
for i in df_mapping.keys():
    if not os.path.exists(get_pickle_name(i,directory_name)): 
        print 'starting '+ i
        print datetime.datetime.now()
        tmp=hive.to_dataframe(df_mapping[i])
        tmp.to_pickle(get_pickle_name(i,directory_name))
        print 'finished '+ i
        print datetime.datetime.now()


starting active_days_df
2018-05-23 11:40:06.590061
finished active_days_df
2018-05-23 12:35:57.858977


In [5]:
start_time = time.time()
results = get_results_parallel(df_mapping, directory_name)
runtime = (time.time() - start_time)


<h1 id="AD-HOC-METRICS">AD HOC METRICS</h1>

<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h1 id="KPI-Metrics">KPI Metrics</h1>

<h2 id="New-Users">New Users</h2><p>Total Users</p>
<p>DAU - Total unique active days between groups</p>
<p>L7 - Total unique days returned after first exposure</p>
<p>Log timespent - Log transformed timespent</p>

<h3 id="Android">Android</h3>

In [7]:
style(pd.concat([
        change_name(df=results, df_name='active_days_df', device='Android', user_type='New', raw='total_users', actions='active_days',
                    raw_new='Total Users', **options),

        change_name(df=results, df_name='active_days_df', device='Android', user_type='New', 
                    raw='daily_active_users', actions='active_days',
                    raw_new='DAU', **options),

        change_name(df=results, df_name='l7_df', device='Android', user_type='New', 
                    raw='l7_per_all_users', actions='l7',
                    raw_new='L7', **options)

       # change_name(df=results, df_name='log_timespent_df', device='Android', user_type='New', 
       #             raw='log_timespent_per_all_users', actions='log_timespent',
       #             raw_new='Log timespent', **options)
]),
    options['variants'], highlight_sig=True)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"Total Users -0.02% 559,265.00 559,175.00"
"DAU 0.7377 -0.07% 32,598.55 0.0041 32,576.86 0.0041"
L7 0.6769 -0.11% 0.35 0.0013 0.35 0.0013


<h2 id="Existing-Users">Existing Users</h2><p>Total Users</p>
<p>DAU - Total unique active days between groups</p>
<p>L7 - Total unique days returned after first exposure</p>
<p>Log timespent - Log transformed timespent</p>

<h3 id="Android">Android</h3>

In [9]:
style(pd.concat([
        change_name(df=results, df_name='active_days_df', device='Android', user_type='Existing', raw='total_users', actions='active_days',
                    raw_new='Total Users', **options),

        change_name(df=results, df_name='active_days_df', device='Android', user_type='Existing', 
                    raw='daily_active_users', actions='active_days',
                    raw_new='DAU', **options),

        change_name(df=results, df_name='l7_df', device='Android', user_type='Existing', 
                    raw='l7_per_all_users', actions='l7',
                    raw_new='L7', **options)#,

       # change_name(df=results, df_name='log_timespent_df', device='Android', user_type='Existing', 
#                     raw='log_timespent_per_all_users', actions='log_timespent',
#                     raw_new='Log timespent', **options)
]
    ),options['variants'], highlight_sig=True)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"Total Users 0.10% 2,409,266.00 2,411,779.00"
"DAU 0.0000 0.80% 842,673.72 0.0133 849,380.24 0.0133"
L7 0.0000 0.14% 0.89 0.0004 0.90 0.0004


<h2 id="Expression">Expression</h2><p>-Original Post</p>
<p>-Original Post DAU</p>
<p>-Original Post DAU (New Users)</p>
<p>-Post with tags (missing)</p>

<h3 id="Android">Android</h3>

In [7]:
style(
    pd.concat([
            change_name(df=results, df_name='posts_df', device='Android', user_type=None, raw='total_posts', actions='posts',
                        raw_new='Original Post', **options),

            change_name(df=results, df_name='posts_df', device='Android', user_type=None, 
                        raw='posts_users', actions='posts',
                        raw_new='Original Post DAU', **options),

          change_name(df=results, df_name='posts_df', device='Android', user_type='New', 
                        raw='posts_users', actions='posts',
                        raw_new='Original Post DAU (New)', **options)]
        ),options['variants'], highlight_sig=True)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"Original Post 2.02% 1,969,290.00 2,009,104.00"
"Original Post DAU 1.61% 779,054.00 791,565.00"
"Original Post DAU (New) -0.88% 48,005.00 47,582.00"


<h2 id="Community">Community</h2><p>-messaging DAU</p>
<p>-messages sent per DAU</p>
<p>-Notes DAU</p>
<p>-Activity DAU</p>
<p>-Notes Views per DAU (missing)</p>

<h3 id="Android">Android</h3>

In [8]:
style(
    pd.concat([
            change_name(df=results, df_name='messages_df', device='Android', user_type=None, 
                        raw='messaging_users', actions='messaging',
                        raw_new='Messagging DAU', **options),
            
            change_name(df=results, df_name='messages_df', device='Android', user_type=None, 
                        raw='messaging_per_all_users', actions='messaging',
                        raw_new='Messagges sent per DAU', **options),
        
        
    
    ]
    ),options['variants'], highlight_sig=True)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"Messagging DAU 2.77% 1,632,936.00 1,678,127.00"
Messagges sent per DAU 0.0000 2.54% 6.54 0.0350 6.71 0.0363


<h2 id="Discovery">Discovery</h2><p>-Likes per DAU</p>
<p>-Reblogs per DAU</p>
<p>-Follows per DAU</p>
<p>-Post Engagment per DAU (missing)</p>
<p>-Reblogs DAU</p>
<p>-Post Impressions (New Users) (Missing)</p>
<p>-Post Impressions (Existing Users) (Missing)</p>

<h3 id="Android">Android</h3>

In [16]:
style(
    pd.concat([
            change_name(df=results, df_name='likes_df', device='Android', user_type=None, 
                        raw='likes_users_pct', actions='likes',
                        raw_new='Likes per DAU', **options),
            
            change_name(df=results, df_name='reblogs_df', device='Android', user_type=None, 
                        raw='reblogs_users_pct', actions='reblogs',
                        raw_new='Reblogs per DAU', **options),
        
            change_name(df=results, df_name='follows_df', device='Android', user_type=None, 
                        raw='follows_users_pct', actions='follows',
                        raw_new='Follows per DAU', **options),
        
            change_name(df=results, df_name='reblogs_df', device='Android', user_type=None, 
                        raw='reblogs_users', actions='reblogs',
                        raw_new='Reblogs DAU', **options),
    ]
    ),options['variants'], highlight_sig=True)


pval1 pct_diff1 control ±(control) test1 ±(test1)
Likes per DAU 1.0000 0.77% 4.02 4.05
Reblogs per DAU 1.0000 1.73% 1.33 1.35
Follows per DAU 1.0000 1.16% 2.12 2.15
"Reblogs DAU 1.84% 3,774,266.00 3,843,789.00"


<h2 id="Push-Notifications">Push Notifications</h2>

<h3 id="Android">Android</h3>

In [23]:
style(
    pd.concat([
            change_name(df=results, df_name='algo_dash_df', device='Android', user_type=None, 
                        raw='aldo_dash_users', actions='aldo_dash',
                        raw_new='Algo Dash DAU', **options),
            
            change_name(df=results, df_name='algo_dash_df', device='Android', user_type=None, 
                        raw='total_aldo_dash', actions='aldo_dash',
                        raw_new='Algo Dash Open', **options),
        
        
            change_name(df=results, df_name='transactional_push_df', device='Android', user_type=None, 
                        raw='transactional_push_users', actions='transactional_push',
                        raw_new='Transactional DAU', **options),
            
            change_name(df=results, df_name='transactional_push_df', device='Android', user_type=None, 
                        raw='total_transactional_push', actions='transactional_push',
                        raw_new='Transactional Opens', **options),
        
        
        
            change_name(df=results, df_name='crm_push_df', device='Android', user_type=None, 
                        raw='crm_push_users', actions='crm_push',
                        raw_new='CRM DAU', **options),
            
            change_name(df=results, df_name='crm_push_df', device='Android', user_type=None, 
                        raw='total_crm_push', actions='crm_push',
                        raw_new='CRM Open', **options)
    ]
    ),options['variants'], highlight_sig=True)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"Algo Dash DAU -0.25% 5,105,824.00 5,093,237.00"
"Algo Dash Open -1.39% 8,092,724.00 7,979,838.00"
"Transactional DAU -0.22% 4,311,598.00 4,301,951.00"
"Transactional Opens -1.18% 27,172,688.00 26,851,512.00"
"CRM DAU -0.42% 42,344.00 42,166.00"
"CRM Open -0.37% 43,207.00 43,048.00"


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h1 id="Appendix">Appendix</h1>

<h2 id="Variant-Count">Variant Count</h2>

In [10]:
variant_count_df = results['variant_count_df']
variant_count_df


,context,users,ct
0,control,6056282,214032358
1,enabled,6061036,215032641


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Total-active-days">Total active days</h2>

In [18]:
active_days_df = filter_df(results['active_days_df'],
                          device='Android',
                          user_type=None)
calc_format(active_days_df,
            actions='active_days',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,839,558.00 2,842,739.00"
"engaging_users 0.11% 2,839,558.00 2,842,739.00"
"active_days_users 0.08% 2,968,531.00 2,970,954.00"
"total_active_days 0.76% 29,540,439.00 29,766,052.00"
active_days_per_active_days_user 0.68% 9.95 10.02
std_active_days 0.35% 10.27 10.31
active_days_per_all_users 0.0000 0.68% 9.95 0.0119 10.02 0.0120
std_active_days_all_users 0.35% 10.27 10.31
"daily_active_users 0.0000 0.76% 875,272.27 0.0119 881,957.10 0.0120"


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="L7">L7</h2><p>Days active <em>in the last 7 full days</em> per user</p>

In [12]:
l7_df = filter_df(results['l7_df'],
                          device='Android',
                          user_type=None)
calc_format(l7_df,
            actions='l7',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 0.35% 1,780,176.00 1,786,346.00"
"l7_users 0.84% 6,570,125.00 6,625,493.00"
"total_l7 0.84% 6,570,125.00 6,625,493.00"
l7_per_l7_user 0.00% 1.00 1.00
std_l7 0.12% 0.00 0.00
l7_per_all_users 0.0000 0.17% 0.85 0.0004 0.85 0.0004
std_l7_all_users -0.40% 0.36 0.36
l7_users_pct 0.73% 2.32 2.33


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Posts">Posts</h2>

In [13]:
posts_df = filter_df(results['posts_df'],
                          device='Android',
                          user_type=None)
calc_format(posts_df,
            actions='posts',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 1.44% 238,488.00 241,927.00"
"posts_users 1.61% 779,054.00 791,565.00"
"total_posts 2.02% 1,969,290.00 2,009,104.00"
posts_per_posts_user 0.2730 0.41% 2.53 0.0128 2.54 0.0133
std_posts 4.61% 5.78 6.05
posts_per_all_users 0.0002 1.69% 0.56 0.0034 0.57 0.0036
std_posts_all_users 4.71% 2.92 3.06
posts_users_pct 0.0000 1.49% 0.27 0.0005 0.28 0.0005


<div align="right"><a href="#toc">Back to contents</a></div><h2 id="Reblogs">Reblogs</h2>

In [14]:
reblogs_df = filter_df(results['reblogs_df'],
                          device='Android',
                          user_type=None)
calc_format(reblogs_df,
            actions='reblogs',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 0.91% 549,324.00 554,325.00"
"reblogs_users 1.84% 3,774,266.00 3,843,789.00"
"total_reblogs 2.91% 28,885,147.00 29,724,803.00"
reblogs_per_reblogs_user 0.0000 1.05% 7.65 0.0179 7.73 0.0179
std_reblogs 1.28% 17.72 17.95
reblogs_per_all_users 0.0000 1.81% 4.67 0.0167 4.76 0.0170
std_reblogs_all_users 1.60% 14.34 14.57
reblogs_users_pct 1.0000 1.73% 1.33 1.35


<p><div align="right"><a href="#toc">Back to contents</a></div>
<a id="lrrp"></a></p>
<h2 id="Likes">Likes</h2>

In [15]:
likes_df = filter_df(results['likes_df'],
                          device='Android',
                          user_type=None)
calc_format(likes_df,
            actions='likes',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 0.33% 1,423,404.00 1,428,033.00"
"likes_users 0.88% 11,389,356.00 11,489,715.00"
"total_likes 1.20% 175,831,783.00 177,940,340.00"
likes_per_likes_user 0.0042 0.32% 15.44 0.0236 15.49 0.0235
std_likes -0.33% 40.72 40.58
likes_per_all_users 0.0676 0.43% 13.63 0.0449 13.69 0.0448
std_likes_all_users -0.27% 38.58 38.48
likes_users_pct 1.0000 0.77% 4.02 4.05


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Replies">Replies</h2>

In [16]:
replies_df = filter_df(results['replies_df'],
                          device='Android',
                          user_type=None)
calc_format(replies_df,
            actions='replies',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 2.12% 136,700.00 139,603.00"
"replies_users 2.12% 136,700.00 139,603.00"
"total_replies -0.55% 923,248.00 918,166.00"
replies_per_replies_user 0.1692 -2.62% 6.75 0.2145 6.58 0.1325
std_replies -37.61% 40.47 25.25
replies_per_all_users 0.7510 -0.63% 0.31 0.0103 0.31 0.0066
std_replies_all_users -35.82% 8.81 5.65
replies_users_pct 0.0000 2.01% 0.05 0.0002 0.05 0.0003


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Messages">Messages</h2>

In [17]:
messages_df = filter_df(results['messages_df'],
                          device='Android',
                          user_type=None)
calc_format(messages_df,
            actions='messaging',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 3.42% 274,062.00 283,435.00"
"messaging_users 2.77% 1,632,936.00 1,678,127.00"
"total_messaging 3.44% 28,258,932.00 29,231,417.00"
messaging_per_messaging_user 0.0303 0.66% 17.31 0.0721 17.42 0.0731
std_messaging 2.84% 47.00 48.34
messaging_per_all_users 0.0000 2.54% 6.54 0.0350 6.71 0.0363
std_messaging_all_users 3.58% 30.10 31.17
messaging_users_pct 0.0000 2.65% 0.58 0.0006 0.59 0.0006


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Follows">Follows</h2>

In [18]:
follows_df = filter_df(results['follows_df'],
                          device='Android',
                          user_type=None)
calc_format(follows_df,
            actions='follows',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 0.51% 1,598,509.00 1,606,675.00"
"follows_users 1.28% 6,018,827.00 6,095,653.00"
"total_follows 1.92% 22,773,543.00 23,211,596.00"
follows_per_follows_user 0.0000 0.64% 3.78 0.0078 3.81 0.0079
std_follows 1.35% 9.79 9.92
follows_per_all_users 0.0001 0.95% 3.11 0.0105 3.13 0.0106
std_follows_all_users 1.47% 8.99 9.12
follows_users_pct 1.0000 1.16% 2.12 2.15


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Unfollows">Unfollows</h2>

In [19]:
unfollows_df = filter_df(results['unfollows_df'],
                          device='Android',
                          user_type=None)
calc_format(unfollows_df,
            actions='unfollows',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 0.73% 558,732.00 562,809.00"
"unfollows_users 1.05% 1,081,650.00 1,093,024.00"
"total_unfollows 2.12% 3,056,563.00 3,121,278.00"
unfollows_per_unfollows_user 0.1442 1.05% 2.83 0.0276 2.86 0.0289
std_unfollows 5.21% 14.66 15.43
unfollows_per_all_users 0.0240 1.84% 0.88 0.0096 0.90 0.0102
std_unfollows_all_users 5.51% 8.28 8.74
unfollows_users_pct 0.0000 0.94% 0.38 0.0006 0.39 0.0006


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Algo-Dash-Push">Algo Dash Push</h2>

In [20]:
algodash_df = filter_df(results['algo_dash_df'],
                          device='Android',
                          user_type=None)
calc_format(algodash_df,
            actions='aldo_dash',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users 0.01% 1,003,251.00 1,003,358.00"
"aldo_dash_users -0.25% 5,105,824.00 5,093,237.00"
"total_aldo_dash -1.39% 8,092,724.00 7,979,838.00"
aldo_dash_per_aldo_dash_user 0.7617 -1.15% 1.58 0.0967 1.57 0.0674
std_aldo_dash -30.41% 111.54 77.62
aldo_dash_per_all_users 0.8338 -1.25% 1.15 0.1106 1.14 0.0769
std_aldo_dash_all_users -30.44% 94.99 66.08
aldo_dash_users_pct 1.0000 -0.36% 1.80 1.79


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="Transactional-Push">Transactional Push</h2>

In [21]:
transactional_df = filter_df(results['transactional_push_df'],
                          device='Android',
                          user_type=None)
calc_format(transactional_df,
            actions='transactional_push',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users -0.15% 548,252.00 547,420.00"
"transactional_push_users -0.22% 4,311,598.00 4,301,951.00"
"total_transactional_push -1.18% 27,172,688.00 26,851,512.00"
transactional_push_per_transactional_push_user 0.4014 -0.96% 6.30 0.1085 6.24 0.0906
std_transactional_push -16.56% 114.95 95.92
transactional_push_per_all_users 0.5377 -1.09% 4.04 0.1073 4.00 0.0894
std_transactional_push_all_users -16.59% 92.13 76.85
transactional_push_users_pct 1.0000 -0.34% 1.52 1.52


<p><div align="right"><a href="#toc">Back to contents</a></div></p>
<h2 id="CRM-Push">CRM Push</h2>

In [22]:
crm_df = filter_df(results['crm_push_df'],
                          device='Android',
                          user_type=None)
calc_format(crm_df,
            actions='crm_push',
            **options)


pval1 pct_diff1 control ±(control) test1 ±(test1)
"total_users 0.11% 2,834,550.00 2,837,773.00"
"engaging_users -0.47% 34,592.00 34,428.00"
"crm_push_users -0.42% 42,344.00 42,166.00"
"total_crm_push -0.37% 43,207.00 43,048.00"
crm_push_per_crm_push_user 0.6296 0.05% 1.02 0.0015 1.02 0.0016
std_crm_push 2.05% 0.16 0.16
crm_push_per_all_users 0.5241 -0.45% 0.01 0.0001 0.01 0.0001
std_crm_push_all_users -0.15% 0.12 0.12
crm_push_users_pct 0.4334 -0.53% 0.01 0.0001 0.01 0.0001
